[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/spinning_top/spinning_top.ipynb)

# A Spinning Top on a Bowl

This example spins a top on a shallow bowl. The top is a heavy disc on a stem with a rounded tip; spun upright with a slight tilt, it does not fall over but swings its axis around the vertical, precessing, while the air slowly takes its spin. Where its tip touches the bowl it is pushed out of the ground and held against sliding by friction.

Every shape is a quadric, a map from points to planes: the top's three parts are solid ellipsoids, and the ground is the solid below a paraboloid. A part's lowest point against the ground is the pole of its tangent plane facing the ground, and how far that point lies below the ground is its depth. The top's inertia is built from a few mass points per part, each joined with its own motion under a rate, `points & points.commutator(Rate)`. The top moves by Lie-group steps of its motor, and after each step every contact is corrected along one force line: pressed out along the ground's normal and held against sliding within the friction cone, each correction the force line's own compliance, `step & body`, with the top's inverse inertia.

In the notation of computational contact mechanics the corrections read as a position-based projection with a Coulomb friction cone, and the shapes as quadric surfaces $\mathbf p^{\mathsf T} Q\,\mathbf p \le 0$ in homogeneous coordinates.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from examples.animation import show_animation
from examples.mechanics.spinning_top import render, scenarios
from examples.mechanics.spinning_top.core import Direction, Plane, Point, Quadric, Rate, mv, w

np.set_printoptions(precision=4, suppress=True)

## 1. The top and the ground

The top's parts are a disc, a stem and a tip, each a solid ellipsoid about its centre, and the ground is a shallow bowl, solid below a paraboloid. Six mass points per part, placed along its axes, carry the part's mass, centroid and second moments. Each mass point joined with its own motion under a rate, `points & points.commutator(Rate)`, is its momentum, and the sum over the points is the top's inertia, a map from rates to forques, with no centre of mass or parallel-axis shift to place.

In [ ]:
# The base top: its spin, tip radius, disc height and mass, the bowl's curvature, the dynamic friction
# and the air's drag; the variations change one of them each.
setup = scenarios.BASE
variations = scenarios.VARIATIONS
# The parts' heights above the tip, their semi-axes and their masses: the disc, the stem and the tip.
part_heights = np.array([setup.disc_height, setup.disc_height + scenarios.STEM_ABOVE_DISC, scenarios.TIP_HEIGHT])
semi_axes = np.stack([scenarios.DISC_SEMI, scenarios.STEM_SEMI, np.array([setup.tip_radius, setup.tip_radius, scenarios.TIP_HEIGHT])])
part_masses = np.array([setup.disc_mass, scenarios.STEM_MASS, scenarios.TIP_MASS])

In [ ]:
# The parts about the top's centre of mass, which sits this high above the tip.
height = (part_heights * part_masses).sum() / part_masses.sum()
mass = part_masses.sum()
centres = np.zeros((3, 3))
centres[:, 2] = part_heights - height
parts = scenarios.core.ellipsoid(mv.point(centres), semi_axes)                 # [parts] Plane <- Point
ground = scenarios.core.bowl(setup.bowl_curvature)                             # [] Plane <- Point

# Six mass points per part; each joined with its own motion under a rate is its momentum, and the sum is
# the inertia.
points, point_masses = scenarios.core.sigma_points(centres, semi_axes, part_masses)   # [points] Point
inertia = (points & points.commutator(Rate) * point_masses).sum()              # [] Forque <- Rate

In [ ]:
print(f"mass {mass:.2f}, centre of mass {height:.3f} above the tip")
print("inertia, from rates to momenta:\n", inertia.coefficient_table())

## 2. The contact point

The ground's form grows towards the air, and a point's polar plane under the ground is the gradient of that form there: read as a direction it is the ground's normal, at any point, on the ground or off it. A part's lowest point against that normal is the pole of its tangent plane on that side, and its depth below the ground is the ground's form there over the gradient's length. Here the top stands in its starting pose, its tip just clear of the bowl.

In [ ]:
def ground_normal(ground: Quadric, points: Point) -> Direction:
    """The ground's unit normal at points, toward the air, where its form grows.

    A point's polar plane under the ground quadric is the gradient of the ground's form there;
    read as a direction and divided by its length it is the normal, at any point, on the ground
    or off it.
    """
    return ground(points).dual().cast(Direction) / ground(points).norm()


def lowest_point(surface: Quadric, normal: Direction) -> Point:
    """Each quadric's point furthest against the normal: the pole of its tangent plane on that side.

    The plane through the quadric's centre normal to the given direction has its pole at infinity:
    the direction conjugate to it, along which the centre reaches the planes tangent to the
    quadric parallel to it. Scaled to reach the surface, it leads from the centre to the lowest
    point. For a part against the ground, the normal is the ground's at the part's centre, so the
    result is the true deepest point only while the ground turns little across the part.
    """
    # Planes to their poles.
    dual = surface.inverse()
    centre = dual(w) / (w & dual(w))
    # The plane through the centre, normal to the direction, and its pole, a direction.
    plane = normal.dual() - w * (normal.dual() & centre)
    conjugate = dual(plane)
    return centre + conjugate / (-(plane & conjugate) * (w & dual(w))).square_root()


# The starting pose: lifted so the tip clears, then tilted.
motor = ((mv.zw * ((height * np.cos(scenarios.TILT) + scenarios.CLEARANCE) / 2)).exp()
         * (mv.zx * (scenarios.TILT / 2)).exp()).normalized()                  # [] Motor
placed = motor >> parts(motor << Point)                                        # [parts] Plane <- Point
centres = placed.solve(w)                                                      # [parts] Point
contacts = lowest_point(placed, ground_normal(ground, centres / (w & centres)))   # [parts] Point
depths = -(ground(contacts) & contacts) / (2 * ground(contacts).norm())        # [parts] Scalar

In [ ]:
print("each part's depth below the ground, disc, stem and tip:", depths.to_array())

## 3. Spinning

From the tilted start the top is stepped by a Lie-group Runge–Kutta step under its weight and the air's drag, and after each step its contacts are corrected. It leans a little further, then holds its lean while its axis swings around the vertical, precessing; the drag slows its spin. The time step is the largest that leaves the motion unchanged. This cell takes about twenty seconds, simulating and drawing the run.

In [ ]:
# The run: its length in seconds, the time step, the steps per frame, and the frames' pixels and duration.
runtime, step, n_steps_per_frame = scenarios.RUNTIME, scenarios.STEP, scenarios.N_STEPS_PER_FRAME
pixels, frame_duration = scenarios.PIXELS, scenarios.FRAME_DURATION

In [ ]:
states = list(scenarios.spin(setup, runtime, step, n_steps_per_frame))         # (motor, rate, parts, ground) per frame
leans = np.array([scenarios.lean(motor).to_array() for motor, _, _, _ in states])   # [frames]
spins = np.array([-(mv.xy | rate).to_array() for _, rate, _, _ in states])     # [frames]
show_animation([render.frame(motor, parts, ground, scenarios.VIEW, scenarios.CENTRE, scenarios.EXTENT, pixels)
                for motor, _, parts, ground in states], "spinning_top_base", frame_duration)

In [ ]:
print(f"lean from {np.degrees(leans[0]):.1f} to at most {np.degrees(leans.max()):.1f} degrees; spin from {spins[0]:.1f} to {spins[-1]:.2f} per second")

## 4. Variations

A sharper tip, a heavier disc and slipperier ground each change one thing about the top. All of them stay up over the run; how far they lean and how much spin they keep differs. This cell takes about forty seconds, simulating every run, the base one included.

In [ ]:
for name, variation in variations.items():
    run = list(scenarios.spin(variation, runtime, step, n_steps_per_frame))
    run_leans = np.array([scenarios.lean(motor).to_array() for motor, _, _, _ in run])
    run_spins = np.array([-(mv.xy | rate).to_array() for _, rate, _, _ in run])
    print(f"{name:10s}: largest lean {np.degrees(run_leans.max()):5.1f} degrees, spin {run_spins[0]:.1f} to {run_spins[-1]:.2f}")

In [ ]:
# checks
# The top starts clear of the ground, stays up over the run, leaning less than halfway over, and keeps
# spinning the same way while the drag slows it.
assert (depths.to_array() <= 0).all()
assert leans.max() < np.pi / 4
assert (spins > 0).all() and spins[-1] < spins[0]